# What our first 13F embedding models actually show

This is a **read-only, locally executed explanation** of aggregate ASMP runs under ignored local model-run directories. Sections 1–5 describe the older prefiltered pilot; Sections 6–7 cover the corrected fit-manager-only runs. It does not fit a model, load individual holdings, display raw identifiers, or save figures. Keep executed notebook outputs local and uncommitted. A colleague with the code but not the private data will see the methods and cautions, not the local scores.

**Question answered:** Given a manager's visible holdings in one quarter, how well does each method assign probability and rank to that manager's hidden second-largest retained position? This is a portfolio-completion/representation diagnostic—not a future-quarter forecast, return prediction, or trading backtest.

## 1. Was this a test or a serious model application?

Both, at different levels. Sections 1–5 describe the **old prefiltered pilot**: real fits on saved point-in-time quarterly 13F matrices, not toy arrays. Managers are split within each quarter, models are fitted, and a held-out manager's rank-two holding is scored. Its 4-dimensional Word2Vec comparison spans the locally available decision-complete quarters; its AssetBERT experiments cover only selected endpoint quarters and two embedding dimensions. Sections 6–7 separately examine the corrected fit-manager-only benchmark, including its expanding 32-dimensional AssetBERT history.

The **old evaluation remains a pilot**. Its stored 13F matrices applied concentration and 20-by-20 degree filters *using all managers before the manager split*. Thus held-out managers could affect eligibility and universe membership, even though their portfolios were not used in parameter fitting. The corrected manager-only runner instead starts from the canonical, prefilter point-in-time state, splits managers first, and freezes filters/vocabulary on fit managers. Neither product has a separate validation split, comparable calibration, repeated seeds, and a held-out final protocol sufficient to establish a best model. The original paper's FactSet fund/company sample and benchmark differ from our 13F manager/typed-security adaptation.

| Established | Not established |
| --- | --- |
| End-to-end model fitting and held-out *manager* queries within a quarter | Leak-free universe construction in these old runs |
| Point-in-time filing-state inputs at each quarterly decision cutoff | Out-of-quarter forecasting or a tradable signal |
| Uniform, popularity, PCA, Word2Vec and limited AssetBERT mechanics | Calibrated or fully tuned model ranking |
| Aggregate likelihood and rank diagnostics | Reproduction of the paper's published FactSet results |

## 2. Who does what?

Each fitted method receives only training-manager portfolios. All scored models face the same masked test managers and the same candidate set: **training vocabulary minus the test manager's visible holdings**. The hidden target is not a visible input.

| Method | Input and learning task | Score at the hidden rank-two slot | Interpretive role |
| --- | --- | --- | --- |
| Uniform | No fit; equal weight on every candidate | Equal logits | Random-choice zero point for normalized ASMP |
| Popularity | Training-manager holding counts | Log(1 + count), identical for all queries | Strong, context-free ownership baseline |
| RS-Binary PCA | Manager × security ownership indicator | Test manager folded into frozen centered PCA loadings using visible holdings | Global co-ownership factor baseline |
| RS-Ranks PCA | Manager × security rank-weighted holdings | Same fold-in, retaining original rank weights after masking | Global factor baseline with within-portfolio ordering |
| Word2Vec / CBOW | Complete ranked training portfolios; predict a token from nearby ranks | Mean embedding of visible rank-window neighbors scored against every asset | Local portfolio-context embedding |
| AssetBERT | Complete ranked training portfolios; masked-token objective | Bidirectional first balanced chunk containing rank two, at most 64 stocks | Contextual portfolio-token prediction |

The PCA scores are used directly as softmax logits and **not temperature-calibrated**. Their normalized likelihood can therefore be disadvantaged by arbitrary score scale, even when rank metrics look competitive. Word2Vec uses a rank-window radius recorded in each run; AssetBERT uses balanced chunks. They do not necessarily observe the same amount of the visible portfolio.

## 3. Load only comparable, aggregate local runs

A run cohort must share its committed model-code state, source commit, code hashes, and complete configuration; it must contain one run per quarter. The selection below never pools differently configured runs. Set `PILOT_COMMIT` to a specific public Git commit if the local cache contains multiple eligible cohorts. No saved run files are modified.

In [ ]:
from collections import defaultdict
import json
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'asmp_13f.py').is_file():
    raise RuntimeError('Run from the asset_embeddings project root or notebooks directory')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RUN_ROOT = PROJECT_ROOT / 'data' / 'model_runs' / '13f_asmp_pilot'
PILOT_PURPOSE = 'provisional_13f_cross_manager_asmp_not_paper_replication'
PILOT_COMMIT = None  # Optional: choose a known committed-code cohort.

def load_aggregate_runs(root):
    if not root.is_dir():
        return []
    loaded = []
    for path in sorted(root.glob('*.json')):
        with path.open(encoding='utf-8') as handle:
            run = json.load(handle)
        if run.get('purpose') == PILOT_PURPOSE and isinstance(run.get('results'), dict):
            loaded.append(run)
    return loaded

runs = load_aggregate_runs(RUN_ROOT)
print('Compatible local run summaries found:', len(runs))

In [ ]:
def cohort_key(run):
    return (
        run['source_code_commit'],
        json.dumps(run['model_code_sha256'], sort_keys=True),
        json.dumps(run['configuration'], sort_keys=True),
    )

eligible = [
    run for run in runs
    if run.get('model_code_committed_at_head') is True
    and run.get('configuration', {}).get('word2vec') is True
    and run['configuration'].get('assetbert') is False
    and run['configuration'].get('neural_dimensions') == 4
    and re.fullmatch(r'\d{4}Q[1-4]', run.get('report_quarter', ''))
    and {'uniform', 'popularity_log_count_plus_one', 'rs_binary', 'rs_ranks', 'word2vec'} <= set(run['results'])
]
if PILOT_COMMIT is not None:
    eligible = [run for run in eligible if run['source_code_commit'] == PILOT_COMMIT]
cohorts = defaultdict(list)
for run in eligible:
    cohorts[cohort_key(run)].append(run)
if not cohorts:
    pilot_runs = []
    selected_commit = selected_code_hashes = selected_configuration = None
    print('No complete old-pilot cohort locally; Sections 3–5 are skipped.')
else:
    largest = max(len(group) for group in cohorts.values())
    leaders = [group for group in cohorts.values() if len(group) == largest]
    if len(leaders) != 1:
        raise ValueError('Multiple equally sized old-pilot cohorts: specify PILOT_COMMIT')
    pilot_runs = sorted(leaders[0], key=lambda run: run['report_quarter'])
    quarters = [run['report_quarter'] for run in pilot_runs]
    if len(quarters) != len(set(quarters)):
        raise ValueError('The selected old-pilot cohort contains duplicate report quarters')
    selected_commit = pilot_runs[0]['source_code_commit']
    selected_code_hashes = pilot_runs[0]['model_code_sha256']
    selected_configuration = pilot_runs[0]['configuration']
    display(pd.DataFrame([{'run_type': '4D Word2Vec + common baselines',
                           'quarters': len(pilot_runs), 'first_quarter': quarters[0],
                           'last_quarter': quarters[-1], 'source_commit': selected_commit[:12],
                           'manager_holdout': selected_configuration['test_manager_fraction'],
                           'seed': selected_configuration['seed'],
                           'Word2Vec_epochs': selected_configuration['w2v_epochs']}]))

The table identifies the run cohort, **not a temporal train/test split**. A fresh manager holdout and a fresh model fit occur separately in each quarter. Repeated managers need not stay on the same side across quarters. Even a complete local quarter range is not evidence of a future-quarter forecast.

## 4. What do the metrics mean?

For each eligible test manager, let $C$ be the training vocabulary after removing visible holdings, and let $p(t)$ be the model's softmax probability for the hidden target. The reported score is

$$\mathrm{ASMP}=1+\frac{\operatorname{mean}[\log p(t)]}{\operatorname{mean}[\log |C|]}. $$

Uniform choice has score 0; 1 would be perfect probability. A positive score means lower average negative log likelihood than uniform **on this candidate set**. It is not classification accuracy, an expected return, or the probability that a trade succeeds.

**Hit@100** is the fraction of hidden targets ranked in the top 100 candidates. **MRR** is mean reciprocal target rank, $\operatorname{mean}(1/\mathrm{rank})$, rewarding very high ranks. They depend on candidate-set size and can favor a different model from likelihood. **Coverage** is the share of test managers *already inside the saved, prefiltered matrix* that have a scoreable query; it is not coverage of the raw 13F product. Out-of-vocabulary and other exclusions must be read alongside every score.

In [ ]:
MODEL_PATHS = {
    'Uniform': ('uniform', None),
    'Popularity': ('popularity_log_count_plus_one', None),
    'RS-Binary 4D': ('rs_binary', '4'),
    'RS-Binary 10D': ('rs_binary', '10'),
    'RS-Ranks 4D': ('rs_ranks', '4'),
    'RS-Ranks 10D': ('rs_ranks', '10'),
    'Word2Vec 4D': ('word2vec', None),
}
METRICS = {
    'ASMP': 'asmp_normalized_log_likelihood',
    'Hit@100': 'hit_at_100',
    'MRR': 'mean_reciprocal_rank',
    'Coverage': 'coverage',
}
rows = []
for run in pilot_runs:
    for model, (family, dimension) in MODEL_PATHS.items():
        result = run['results'][family]
        if dimension is not None:
            result = result[dimension]
        row = {'quarter': run['report_quarter'], 'model': model}
        row.update({label: float(result[key]) for label, key in METRICS.items()})
        row.update({'input_portfolios': int(result['input_portfolios']),
                    'scored_portfolios': int(result['scored_portfolios']),
                    'OOV_exclusions': int(result['excluded_out_of_vocabulary'])})
        rows.append(row)
results_local = pd.DataFrame(rows)
if not results_local.empty:
    if results_local.groupby('quarter')['scored_portfolios'].nunique().ne(1).any():
        raise ValueError('Models in a quarter were not scored on identical query counts')
    if results_local.groupby('quarter')['Coverage'].nunique().ne(1).any():
        raise ValueError('Models in a quarter had different coverage')
    summary = results_local.groupby('model', sort=False)[list(METRICS)].median().round(3)
    display(summary.rename_axis('Model (quarter median)'))
    display(results_local.loc[results_local.model.eq('Word2Vec 4D'),
                              ['quarter', 'input_portfolios', 'scored_portfolios',
                               'OOV_exclusions', 'Coverage']].reset_index(drop=True))

The medians treat each quarter equally; they do not pool all test managers into one grand likelihood. The second table makes eligibility visible. Do not interpret 100% within-matrix coverage, if observed, as 100% coverage of unfiltered filings.

In [ ]:
if not results_local.empty:
    models = list(MODEL_PATHS)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    for axis, metric in zip(axes, ('ASMP', 'Hit@100', 'MRR')):
        distributions = [results_local.loc[results_local.model.eq(model), metric].to_numpy()
                         for model in models]
        axis.boxplot(distributions, tick_labels=models, showfliers=False)
        axis.set_title(metric + ' across separately fitted quarters')
        axis.tick_params(axis='x', rotation=65)
        axis.grid(axis='y', alpha=0.25)
    axes[0].axhline(0, color='black', linewidth=0.8, alpha=0.5)
    plt.show()

These boxplots are *quarter distributions*, not confidence intervals or out-of-sample time-series performance. A favorable median across the selected run cohort does not rank methods definitively: the vocabulary was prefiltered, PCA logits are uncalibrated, and model budgets differ.

In [ ]:
if not results_local.empty:
    paired = results_local.pivot(index='quarter', columns='model', values=['ASMP', 'Hit@100', 'MRR'])
    paired_deltas = pd.DataFrame({
        metric: paired[(metric, 'Word2Vec 4D')] - paired[(metric, 'Popularity')]
        for metric in ('ASMP', 'Hit@100', 'MRR')
    })
    display(pd.DataFrame({'median_Word2Vec_minus_popularity': paired_deltas.median(),
                          'quarters_above_popularity': paired_deltas.gt(0).sum(),
                          'quarters_compared': len(paired_deltas)}).round(3))
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
    for axis, metric in zip(axes, paired_deltas.columns):
        axis.plot(range(len(paired_deltas)), paired_deltas[metric], marker='.', linewidth=1)
        axis.axhline(0, color='black', linewidth=0.8)
        axis.set_ylabel(metric + ' difference')
        axis.grid(axis='y', alpha=0.25)
    axes[-1].set_xticks(range(0, len(paired_deltas), max(1, len(paired_deltas) // 10)),
                       list(paired_deltas.index)[::max(1, len(paired_deltas) // 10)], rotation=45)
    axes[-1].set_xlabel('Report quarter; each point is a new within-quarter fit and test')
    plt.show()

## 5. AssetBERT: separate endpoint experiments

These **old prefiltered-pilot AssetBERT** runs are feasibility tests on selected endpoint quarters; they are not the corrected full-history comparison in Section 7. Compare Word2Vec and AssetBERT within the same quarter and dimension only; do not merge 4D and 32D rows. The runner records their epochs separately. Different objectives, context windows, and training budgets preclude an untuned winner claim. This section requires a matching source commit, code hashes, matrix hash, and held-out counts.

In [ ]:
pilot_by_quarter = {run['report_quarter']: run for run in pilot_runs}
bert_runs = []
for run in runs:
    config = run.get('configuration', {})
    quarter = run.get('report_quarter')
    if not pilot_runs or not (run.get('model_code_committed_at_head') is True
            and run.get('source_code_commit') == selected_commit
            and run.get('model_code_sha256') == selected_code_hashes
            and config.get('word2vec') is True
            and config.get('assetbert') is True
            and {'word2vec', 'assetbert', 'popularity_log_count_plus_one'} <= set(run['results'])
            and quarter in pilot_by_quarter):
        continue
    baseline = pilot_by_quarter[quarter]
    if (run.get('source_pair_file_sha256') != baseline.get('source_pair_file_sha256')
            or run.get('fitted_vocabulary_sha256') != baseline.get('fitted_vocabulary_sha256')
            or run.get('test_manager_count') != baseline.get('test_manager_count')
            or config.get('seed') != selected_configuration['seed']
            or config.get('test_manager_fraction') != selected_configuration['test_manager_fraction']):
        raise ValueError('AssetBERT and baseline provenance or manager split disagree')
    bert_runs.append(run)
if not bert_runs:
    print('No matching local AssetBERT endpoint runs are available.')
else:
    endpoint_rows = []
    seen = set()
    for run in sorted(bert_runs, key=lambda item: (item['report_quarter'], item['configuration']['neural_dimensions'])):
        quarter = run['report_quarter']
        dim = run['configuration']['neural_dimensions']
        if (quarter, dim) in seen:
            raise ValueError('Duplicate AssetBERT endpoint configuration')
        seen.add((quarter, dim))
        for name, key in (('Popularity', 'popularity_log_count_plus_one'),
                          ('Word2Vec', 'word2vec'), ('AssetBERT', 'assetbert')):
            result = run['results'][key]
            endpoint_rows.append({'quarter': quarter, 'dimensions': dim, 'model': name,
                                  'ASMP': float(result['asmp_normalized_log_likelihood']),
                                  'Hit@100': float(result['hit_at_100']),
                                  'MRR': float(result['mean_reciprocal_rank']),
                                  'scored': int(result['scored_portfolios']),
                                  'W2V_epochs': run['configuration']['w2v_epochs'],
                                  'BERT_epochs': run['configuration']['bert_epochs']})
    endpoints_local = pd.DataFrame(endpoint_rows)
    if endpoints_local.groupby(['quarter', 'dimensions'])['scored'].nunique().ne(1).any():
        raise ValueError('Endpoint models did not score the same test-manager count')
    display(endpoints_local.round(3).reset_index(drop=True))
    for dim, panel in endpoints_local.groupby('dimensions'):
        pivot = panel.pivot(index='quarter', columns='model', values='ASMP')
        ax = pivot[['Popularity', 'Word2Vec', 'AssetBERT']].plot.bar(
            figsize=(8, 4), rot=0, title=f'Endpoint ASMP, {dim} dimensions')
        ax.set_ylabel('Normalized log-likelihood')
        ax.grid(axis='y', alpha=0.25)
        plt.show()

## 6. Separate fit-manager-only universe benchmark (if available)

This optional section reads a **different** ignored run product, built from the prefilter point-in-time 13F state. The runner assigns managers before any fit-side concentration or degree filter; only fit managers determine that universe. It then tests untouched managers while reporting rank-two out-of-vocabulary exclusions and later visible positions dropped from the query. This repairs the old pilot's *test-manager influence on universe construction*, but it does **not** establish a temporal forecast, fair hyperparameter selection, or the paper's original FactSet result.

The charts below compare models **within one selected manager-only run only**. Do not subtract or compare their numeric levels directly against Sections 3–5: the universe, eligibility denominator, and candidate sets changed. If there are multiple committed-code runs for a quarter, set `MANAGER_ONLY_RUN_FILE` to the exact local aggregate JSON filename; the notebook will not silently choose among configurations.

In [ ]:
MANAGER_ONLY_ROOT = PROJECT_ROOT / 'data' / 'model_runs' / '13f_manager_only_asmp'
MANAGER_ONLY_PURPOSE = '13f_cross_manager_fit_only_universe_not_temporal_or_paper_replication'
MANAGER_ONLY_QUARTER = None  # Optional YYYYQX selector.
MANAGER_ONLY_SOURCE_COMMIT = None  # Optional full public Git commit hash.
MANAGER_ONLY_RUN_FILE = None  # Optional exact filename when configurations overlap.
manager_only_runs = []
if MANAGER_ONLY_ROOT.is_dir():
    for path in sorted(MANAGER_ONLY_ROOT.glob('*.json')):
        if MANAGER_ONLY_RUN_FILE is not None and path.name != MANAGER_ONLY_RUN_FILE:
            continue
        with path.open(encoding='utf-8') as handle:
            candidate = json.load(handle)
        if candidate.get('purpose') != MANAGER_ONLY_PURPOSE:
            continue
        if candidate.get('model_code_committed_at_head') is not True:
            continue
        if not (re.fullmatch(r'[0-9a-f]{40}', candidate.get('source_code_commit', ''))
                and isinstance(candidate.get('model_code_sha256'), dict)
                and candidate['model_code_sha256']
                and all(isinstance(digest, str) and re.fullmatch(r'[0-9a-f]{64}', digest)
                        for digest in candidate['model_code_sha256'].values())
                and candidate.get('input_fingerprint')
                and isinstance(candidate.get('configuration'), dict)
                and isinstance(candidate.get('results'), dict)
                and candidate.get('eligibility')):
            continue
        if (MANAGER_ONLY_QUARTER is not None
                and candidate.get('report_quarter') != MANAGER_ONLY_QUARTER):
            continue
        if (MANAGER_ONLY_SOURCE_COMMIT is not None
                and candidate['source_code_commit'] != MANAGER_ONLY_SOURCE_COMMIT):
            continue
        candidate['_local_filename'] = path.name  # For explicit run selection only.
        manager_only_runs.append(candidate)
if not manager_only_runs:
    manager_only_run = None
    print('No committed-code fit-manager-only aggregate run is available locally yet.')
else:
    latest_quarter = max(run['report_quarter'] for run in manager_only_runs)
    matching = [run for run in manager_only_runs if run['report_quarter'] == latest_quarter]
    if len(matching) != 1:
        manager_only_run = None
        print('Multiple committed-code runs match the latest selected quarter; set MANAGER_ONLY_RUN_FILE before charting.')
        display(pd.DataFrame([{'quarter': run['report_quarter'],
                               'local_filename': run['_local_filename'],
                               'source_commit': run['source_code_commit'][:12],
                               'dimensions': run['configuration'].get('neural_dimensions'),
                               'Word2Vec': run['configuration'].get('word2vec'),
                               'AssetBERT': run['configuration'].get('assetbert')}
                              for run in matching]))
    else:
        manager_only_run = matching[0]
        display(pd.DataFrame([{'quarter': manager_only_run['report_quarter'],
                               'source_commit': manager_only_run['source_code_commit'][:12],
                               'manager_split': manager_only_run['manager_split_policy'],
                               'seed': manager_only_run['configuration']['seed'],
                               'dimensions': manager_only_run['configuration']['neural_dimensions'],
                               'Word2Vec': manager_only_run['configuration']['word2vec'],
                               'AssetBERT': manager_only_run['configuration']['assetbert']}]))

In [ ]:
if manager_only_run is not None:
    run = manager_only_run
    metrics = run['results']
    names = {'uniform': 'Uniform', 'popularity_log_count_plus_one': 'Popularity',
             'rs_binary': 'RS-Binary', 'rs_ranks': 'RS-Ranks',
             'word2vec': 'Word2Vec', 'assetbert': 'AssetBERT'}
    model_rows = []
    for family, value in metrics.items():
        if family not in names:
            continue
        versions = value.items() if family in ('rs_binary', 'rs_ranks') else [(None, value)]
        for dimension, result in versions:
            if result.get('benchmark') != MANAGER_ONLY_PURPOSE:
                raise ValueError('A result belongs to a different benchmark contract')
            label = names[family] + (f' {dimension}D' if dimension else '')
            model_rows.append({'model': label,
                               'ASMP': float(result['asmp_normalized_log_likelihood']),
                               'Hit@100': float(result['hit_at_100']),
                               'MRR': float(result['mean_reciprocal_rank']),
                               'scored': int(result['scored_portfolios']),
                               'input': int(result['input_portfolios']),
                               'coverage_within_positive_test_portfolios': float(result['coverage']),
                               'OOV_exclusions': int(result['excluded_out_of_vocabulary'])})
    if not model_rows:
        raise ValueError('The selected manager-only run has no supported model metrics')
    manager_only_metrics_local = pd.DataFrame(model_rows).set_index('model')
    if manager_only_metrics_local[['scored', 'input', 'OOV_exclusions']].nunique().gt(1).any():
        raise ValueError('Manager-only models did not share the same test queries')
    display(manager_only_metrics_local.round(3))
    baseline = metrics['uniform']
    eligibility = run['eligibility']
    assigned_test = int(eligibility['assigned_test_managers'])
    positive_test = int(eligibility['test_managers_with_positive_positions'])
    entered = int(baseline['input_portfolios'])
    scored = int(baseline['scored_portfolios'])
    exclusions = sum(int(baseline[key]) for key in ('excluded_short_portfolios',
                      'excluded_out_of_vocabulary', 'excluded_too_few_candidates'))
    if not (assigned_test > 0 and assigned_test >= positive_test >= entered >= scored
            and entered - scored == exclusions
            and int(eligibility['assigned_fit_managers']) + assigned_test == int(eligibility['eligible_manager_count'])):
        raise ValueError('Manager-only scored count does not reconcile to assigned test managers')
    fit_side_local = pd.DataFrame([
        {'stage': 'PIT-eligible managers (both partitions)', 'count': eligibility['eligible_manager_count']},
        {'stage': 'Assigned fit managers', 'count': eligibility['assigned_fit_managers']},
        {'stage': 'Fit managers surviving fit-only filters', 'count': eligibility['fit_managers_after_filters']},
    ])
    test_attrition_local = pd.DataFrame([
        {'stage': 'Assigned test managers', 'count': assigned_test},
        {'stage': 'Test managers with positive eligible positions', 'count': positive_test},
        {'stage': 'Test portfolios entering masked-query eligibility', 'count': entered},
        {'stage': 'Scored test managers', 'count': scored},
    ])
    exclusions_local = pd.DataFrame([
        {'stage': 'Excluded: short portfolios', 'count': baseline['excluded_short_portfolios']},
        {'stage': 'Excluded: rank-two/out-of-vocabulary', 'count': baseline['excluded_out_of_vocabulary']},
        {'stage': 'Excluded: too few candidates', 'count': baseline['excluded_too_few_candidates']},
])
    print('Fit-side provenance')
    display(fit_side_local)
    print('Test-manager attrition ladder')
    display(test_attrition_local)
    display(exclusions_local)
    display(pd.DataFrame([{'assigned_test_to_scored_share': scored / assigned_test,
                           'within_positive_test_portfolio_coverage': baseline['coverage'],
                           'later_visible_OOV_positions_dropped': eligibility['dropped_oov_visible_positions'],
                           'test_pairs_before_projection': eligibility['test_pairs_before_universe_projection']}]).round(3))
    axes = manager_only_metrics_local[['ASMP', 'Hit@100', 'MRR']].plot.bar(
        subplots=True, layout=(1, 3), figsize=(15, 4), legend=False, rot=65,
        title='Within-run manager-only comparison; distinct metric scales')
    for axis in np.asarray(axes).ravel():
        axis.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

The **assigned-test-to-scored share** uses all managers assigned to the test partition; it is intentionally broader than the model's `coverage`, whose denominator is positive-position portfolios entering query eligibility. The test attrition ladder must be read in order. OOV exclusions include a missing top holding or missing original rank-two target. Later OOV *visible* positions are instead dropped and counted; this compresses the remaining rank context and can change RS-Ranks weights even though the original rank-two target stays fixed. Neither coverage measure should be imported into the older prefiltered-pilot charts.

## 7. Corrected 32D Word2Vec and AssetBERT across available quarters

This read-only section compares the fit-manager-only runs with seed 17, 32 dimensions, one Word2Vec epoch, and five AssetBERT epochs. Each quarter has its own manager split and model fit. Expected quarters come from the same raw local 13F product used by the runner, not the older saved all-manager matrices. If that product is unavailable, full-history coverage is **unverified**. Only one cohort with identical committed model-code hashes, full configuration, raw-product fingerprint, and split/mask policies plus matching within-run test contracts can be compared. Distinct source commits are displayed because documentation-only commits need not change the model code. Missing or ambiguous quarters remain visible while the batch is incomplete.

Set FULL_HISTORY_COHORT_FILE to an exact ignored local filename only if multiple valid cohorts exist. Invalid files from other or unattributed cohorts appear as separate warnings; duplicates or invalid files attributable to the selected cohort block completion. Other dimensions and the older prefiltered pilot are not pooled here.

In [ ]:
from src.matrix_13f import decision_complete_quarters, product_input_fingerprint

FULL_HISTORY_PRODUCT_ROOT = PROJECT_ROOT / 'data' / 'full_history' / '13f' / 'v1'
FULL_HISTORY_COHORT_FILE = None  # Exact ignored JSON filename only when cohorts overlap.
FULL_HISTORY_PROTOCOL = {
    'seed': 17, 'test_manager_fraction': 0.2,
    'fit_universe_filters': {'maximum_position_weight': 0.75,
                             'minimum_assets_per_manager': 20,
                             'minimum_managers_per_asset': 20, 'batch_size': 250000},
    'pca_dimensions': [4, 10], 'word2vec': True, 'assetbert': True,
    'neural_dimensions': 32, 'w2v_radius': 5, 'w2v_epochs': 1,
    'w2v_batch_size': 128, 'w2v_learning_rate': 0.01,
    'bert_epochs': 5, 'bert_batch_size': 32, 'bert_learning_rate': 0.001,
}
expected_by_quarter = {}
current_input_fingerprint = None
if FULL_HISTORY_PRODUCT_ROOT.is_dir():
    coverage = decision_complete_quarters(FULL_HISTORY_PRODUCT_ROOT)
    for row in coverage.itertuples(index=False):
        report_date = pd.to_datetime(row.report_date_utc, utc=True)
        quarter = str(report_date.tz_localize(None).to_period('Q'))
        if quarter in expected_by_quarter:
            raise ValueError('Duplicate decision-complete quarter in raw 13F product')
        expected_by_quarter[quarter] = (report_date,
            pd.to_datetime(row.information_cutoff_utc, utc=True))
    current_input_fingerprint = product_input_fingerprint(FULL_HISTORY_PRODUCT_ROOT)
else:
    print('Raw 13F product unavailable: full-history coverage is unverified.')

full_history_candidates = []
incomplete_json_quarters = set()
if MANAGER_ONLY_ROOT.is_dir():
    for path in sorted(MANAGER_ONLY_ROOT.glob('*.json')):
        try:
            with path.open(encoding='utf-8') as handle:
                run = json.load(handle)
        except json.JSONDecodeError:
            if re.match(r'^\d{4}Q[1-4]_', path.name):
                incomplete_json_quarters.add(path.name[:6])
            continue  # A concurrently written JSON is not a completed run.
        if not isinstance(run, dict):
            if re.match(r'^\d{4}Q[1-4]_', path.name):
                incomplete_json_quarters.add(path.name[:6])
            continue
        if run.get('purpose') != MANAGER_ONLY_PURPOSE:
            continue
        config = run.get('configuration', {})
        if not isinstance(config, dict) or any(
                config.get(key) != value for key, value in FULL_HISTORY_PROTOCOL.items()):
            continue
        run['_local_filename'] = path.name
        full_history_candidates.append(run)

CONTRACT_FIELDS = ('benchmark', 'masked_rank', 'candidate_policy', 'vocabulary_size',
    'input_portfolios', 'scored_portfolios', 'coverage', 'excluded_short_portfolios',
    'excluded_out_of_vocabulary', 'excluded_too_few_candidates',
    'dropped_out_of_vocabulary_visible_positions', 'mean_log_random_probability')
def full_history_issue(run):
    quarter = run.get('report_quarter', '')
    if not isinstance(quarter, str) or not re.fullmatch(r'\d{4}Q[1-4]', quarter):
        return 'invalid quarter label'
    if run.get('model_code_committed_at_head') is not True:
        return 'model code uncommitted at run time'
    if not isinstance(run.get('source_code_commit'), str) or not re.fullmatch(
            r'[0-9a-f]{40}', run['source_code_commit']):
        return 'invalid source commit'
    hashes = run.get('model_code_sha256')
    if not isinstance(hashes, dict) or not hashes or any(
            not isinstance(value, str) or not re.fullmatch(r'[0-9a-f]{64}', value)
            for value in hashes.values()):
        return 'invalid model-code hashes'
    for key in ('input_fingerprint', 'fit_pair_content_sha256',
                'test_pair_content_sha256', 'fitted_vocabulary_sha256'):
        if not isinstance(run.get(key), str) or not re.fullmatch(r'[0-9a-f]{64}', run[key]):
            return 'invalid data or split fingerprint'
    if run.get('input_fingerprint_method') != 'relative_path_size_mtime_cache_signal_not_content_checksum':
        return 'unexpected raw-product fingerprint method'
    if current_input_fingerprint and run['input_fingerprint'] != current_input_fingerprint:
        return 'stale raw-product fingerprint'
    if (run.get('manager_split_policy') != 'PIT-eligible manager IDs split before fit-side universe filters'
            or run.get('masked_edge_policy') !=
            'original_second_largest_positive_eligible_cash_share_position; ranks_one_and_two_require_fit_vocabulary; later_OOV_visible_positions_dropped'):
        return 'unexpected manager split or masked-edge policy'
    if current_input_fingerprint and quarter not in expected_by_quarter:
        return 'outside decision-complete history'
    try:
        if quarter in expected_by_quarter:
            report, cutoff = expected_by_quarter[quarter]
            if (pd.to_datetime(run['report_date_utc'], utc=True) != report
                    or pd.to_datetime(run['decision_at_utc'], utc=True) != cutoff):
                return 'report date or decision cutoff differs from raw product'
        results = run['results']
        variants = [results['uniform'], results['popularity_log_count_plus_one'],
                    results['rs_binary']['4'], results['rs_binary']['10'],
                    results['rs_ranks']['4'], results['rs_ranks']['10'],
                    results['word2vec'], results['assetbert']]
        contracts = [tuple(item[key] for key in CONTRACT_FIELDS) for item in variants]
        if len(set(contracts)) != 1:
            return 'within-run test contracts or exclusions differ'
        base = variants[0]
        if (base['benchmark'] != MANAGER_ONLY_PURPOSE or base['masked_rank'] != 2
                or base['candidate_policy'] != 'frozen_vocabulary_minus_visible_holdings'):
            return 'unexpected benchmark or candidate policy'
        e = run['eligibility']
        assigned, entered, scored = (int(e['assigned_test_managers']),
            int(base['input_portfolios']), int(base['scored_portfolios']))
        excluded = sum(int(base[key]) for key in ('excluded_short_portfolios',
            'excluded_out_of_vocabulary', 'excluded_too_few_candidates'))
        if not (assigned > 0 and assigned >= int(e['test_managers_with_positive_positions'])
                >= entered >= scored > 0 and entered - scored == excluded
                and int(e['test_managers_with_positive_positions']) == int(run['test_manager_count'])
                and int(e['fit_managers_after_filters']) == int(run['fit_manager_count'])
                and int(e['assigned_fit_managers']) + assigned == int(e['eligible_manager_count'])
                and int(e['dropped_oov_visible_positions'])
                == int(base['dropped_out_of_vocabulary_visible_positions'])):
            return 'manager/test/OOV counts do not reconcile'
        w2v_fit, bert_fit, config = (results['word2vec']['fit'],
            results['assetbert']['fit'], run['configuration'])
        if (w2v_fit['dimensions'] != 32 or w2v_fit['epochs'] != 1
                or w2v_fit['rank_radius'] != config['w2v_radius']
                or w2v_fit['batch_size'] != config['w2v_batch_size']
                or w2v_fit['learning_rate'] != config['w2v_learning_rate']
                or w2v_fit['training_managers'] != run['fit_manager_count']
                or bert_fit['dimensions'] != 32 or bert_fit['epochs'] != 5
                or bert_fit['batch_size'] != config['bert_batch_size']
                or bert_fit['learning_rate'] != config['bert_learning_rate']
                or bert_fit['training_managers'] != run['fit_manager_count']
                or (config.get('bert_device') in ('cpu', 'mps')
                    and bert_fit['device'] != config['bert_device'])):
            return 'recorded neural fits differ from protocol'
        if any(not np.isfinite(float(item[key])) for item in variants
               for key in ('asmp_normalized_log_likelihood', 'hit_at_100',
                           'mean_reciprocal_rank')):
            return 'non-finite model score'
    except (KeyError, TypeError, ValueError, OverflowError):
        return 'malformed result or eligibility contract'
    return None

valid_full_history_runs = []
rejected_candidates = []
for run in full_history_candidates:
    issue = full_history_issue(run)
    if issue:
        rejected_candidates.append((run, issue))
    else:
        valid_full_history_runs.append(run)
def full_history_cohort_key(run):
    return (json.dumps(run['model_code_sha256'], sort_keys=True),
            json.dumps(run['configuration'], sort_keys=True),
            run['input_fingerprint'], run['input_fingerprint_method'],
            run['manager_split_policy'],
            run['masked_edge_policy'])
full_history_cohorts = defaultdict(list)
for run in valid_full_history_runs:
    full_history_cohorts[full_history_cohort_key(run)].append(run)
display(pd.DataFrame([{'source_commits': ', '.join(sorted({
                           run['source_code_commit'][:12] for run in group})),
                       'quarters': len({run['report_quarter'] for run in group}),
                       'files': len(group),
                       'BERT_device': group[0]['configuration'].get('bert_device')}
                      for key, group in full_history_cohorts.items()]))
selected_full_history_key = None
if FULL_HISTORY_COHORT_FILE is not None:
    anchors = [run for run in valid_full_history_runs
               if run['_local_filename'] == FULL_HISTORY_COHORT_FILE]
    if len(anchors) != 1:
        raise ValueError('FULL_HISTORY_COHORT_FILE must name one valid matching-protocol run')
    selected_full_history_key = full_history_cohort_key(anchors[0])
elif len(full_history_cohorts) == 1:
    selected_full_history_key = next(iter(full_history_cohorts))
elif len(full_history_cohorts) > 1:
    print('Multiple valid cohorts: set FULL_HISTORY_COHORT_FILE; scores are not pooled.')
else:
    print('No valid matching-protocol run is available locally yet.')
selected_full_history_runs = (full_history_cohorts[selected_full_history_key]
                              if selected_full_history_key is not None else [])
selected_cohort_rejections = defaultdict(list)
warning_rows = []
for run, issue in rejected_candidates:
    try:
        same_cohort = full_history_cohort_key(run) == selected_full_history_key
    except (KeyError, TypeError, ValueError):
        same_cohort = False  # Broken provenance cannot be attributed to a cohort.
    if same_cohort and selected_full_history_key is not None:
        selected_cohort_rejections[run.get('report_quarter', '?')].append(issue)
    warning_rows.append({'quarter': run.get('report_quarter', '?'),
                         'issue': issue, 'selected_cohort': same_cohort})
for quarter in sorted(incomplete_json_quarters):
    warning_rows.append({'quarter': quarter, 'issue': 'incomplete JSON, possibly still writing',
                         'selected_cohort': False})
if warning_rows:
    print('Local file warnings (unattributed or other-cohort files do not change selected-cohort status):')
    display(pd.DataFrame(warning_rows))

In [ ]:
selected_by_quarter = defaultdict(list)
for run in selected_full_history_runs:
    selected_by_quarter[run['report_quarter']].append(run)
quarter_scope = (sorted(expected_by_quarter) if current_input_fingerprint
                 else sorted({run['report_quarter'] for run in full_history_candidates
                              if isinstance(run.get('report_quarter'), str)
                              and re.fullmatch(r'\d{4}Q[1-4]', run['report_quarter'])}))
status_rows = []
for quarter in quarter_scope:
    matches = selected_by_quarter[quarter]
    if selected_full_history_key is None and len(full_history_cohorts) > 1:
        status = 'cohort selection required'
    elif len(matches) > 1:
        status = 'ambiguous duplicate runs'
    elif selected_cohort_rejections.get(quarter):
        status = 'invalid file in selected cohort'
    elif len(matches) == 1:
        status = 'ready'
    else:
        status = 'missing selected-cohort run'
    status_rows.append({'quarter': quarter, 'status': status,
                        'selected_cohort_issue': '; '.join(sorted(set(selected_cohort_rejections.get(quarter, [])))),
                        'selected_cohort_files': len(matches)})
full_history_status_local = pd.DataFrame(status_rows)
complete_full_history = bool(current_input_fingerprint and status_rows
    and len(status_rows) == len(expected_by_quarter)
    and full_history_status_local['status'].eq('ready').all())
print('Full-history status:', 'COMPLETE, locally source-verified' if complete_full_history
      else 'INCOMPLETE or source coverage unverified')
if not full_history_status_local.empty:
    display(full_history_status_local['status'].value_counts().rename_axis('status')
            .reset_index(name='quarters'))
    display(full_history_status_local.loc[full_history_status_local['status'].ne('ready')])
ready_quarters = set(full_history_status_local.loc[
    full_history_status_local['status'].eq('ready'), 'quarter']) if status_rows else set()
ready_runs = [selected_by_quarter[quarter][0] for quarter in sorted(ready_quarters)]

score_rows, coverage_rows = [], []
for run in ready_runs:
    quarter, models = run['report_quarter'], run['results']
    row = {'quarter': quarter}
    for label, key in (('Popularity', 'popularity_log_count_plus_one'),
                       ('Word2Vec', 'word2vec'), ('AssetBERT', 'assetbert')):
        result = models[key]
        row[f'{label} ASMP'] = float(result['asmp_normalized_log_likelihood'])
        row[f'{label} Hit@100'] = float(result['hit_at_100'])
        row[f'{label} MRR'] = float(result['mean_reciprocal_rank'])
    for metric in ('ASMP', 'Hit@100', 'MRR'):
        row[f'BERT minus W2V {metric}'] = row[f'AssetBERT {metric}'] - row[f'Word2Vec {metric}']
    score_rows.append(row)
    base = models['uniform']
    assigned = int(run['eligibility']['assigned_test_managers'])
    coverage_rows.append({'quarter': quarter, 'assigned_test': assigned,
                          'scored': int(base['scored_portfolios']),
                          'scored_share_assigned': int(base['scored_portfolios']) / assigned,
                          'coverage_within_positive': float(base['coverage']),
                          'rank1_or_rank2_OOV_exclusions': int(base['excluded_out_of_vocabulary']),
                          'later_visible_OOV_positions_dropped': int(run['eligibility']['dropped_oov_visible_positions']),
                          'fitted_vocabulary_size': int(base['vocabulary_size'])})
full_history_scores_local = pd.DataFrame(score_rows).set_index('quarter') if score_rows else pd.DataFrame()
full_history_coverage_local = pd.DataFrame(coverage_rows).set_index('quarter') if coverage_rows else pd.DataFrame()
if not full_history_scores_local.empty:
    display(full_history_scores_local.round(3))
    display(full_history_coverage_local.round(3))
    display(pd.DataFrame([{'metric': metric, 'quarters': len(full_history_scores_local),
                           'median_BERT': full_history_scores_local[f'AssetBERT {metric}'].median(),
                           'median_W2V': full_history_scores_local[f'Word2Vec {metric}'].median(),
                           'median_BERT_minus_W2V': full_history_scores_local[f'BERT minus W2V {metric}'].median(),
                           'BERT_above_W2V_quarters': int(full_history_scores_local[f'BERT minus W2V {metric}'].gt(0).sum())}
                          for metric in ('ASMP', 'Hit@100', 'MRR')]).round(3))

In [ ]:
if not full_history_scores_local.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
    plot_scores = full_history_scores_local.reindex(quarter_scope)
    # Actual quarter-end dates preserve elapsed time; missing expected quarters remain NaN gaps.
    plot_dates = pd.PeriodIndex(plot_scores.index, freq='Q').to_timestamp(how='end').normalize()
    tick_positions = np.arange(0, len(plot_scores), max(1, len(plot_scores) // 12))
    for ax, metric in zip(axes.flat[:3], ('ASMP', 'Hit@100', 'MRR')):
        for model in ('Popularity', 'Word2Vec', 'AssetBERT'):
            ax.plot(plot_dates, plot_scores[f'{model} {metric}'],
                    marker='.', linewidth=1.3, label=model)
        ax.set_title(metric + ' by report quarter')
        ax.set_ylabel({'ASMP': 'Normalized log-likelihood gain',
                       'Hit@100': 'Share recovered in top 100',
                       'MRR': 'Mean reciprocal rank'}[metric])
        ax.set_xticks(plot_dates[tick_positions], plot_scores.index[tick_positions], rotation=55)
        ax.grid(alpha=0.25)
        ax.legend()
    ax = axes.flat[3]
    ax.plot(plot_dates, plot_scores['BERT minus W2V ASMP'], marker='.', color='purple')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('Paired ASMP difference: AssetBERT minus Word2Vec')
    ax.set_ylabel('BERT minus W2V normalized likelihood gain')
    ax.set_xticks(plot_dates[tick_positions], plot_scores.index[tick_positions], rotation=55)
    ax.grid(alpha=0.25)
    plt.show()

These are **within-quarter cross-manager** portfolio-completion diagnostics, not chronological train/test forecasts. ASMP measures normalized log-likelihood improvement over uniform, not classification accuracy or a return. The quarterly median weights each quarter equally and is not a pooled likelihood or uncertainty interval. The coverage table distinguishes scored/assigned managers from coverage conditional on positive eligible positions; rank-one/rank-two OOV exclusions differ from later visible OOV positions dropped from context. The raw-product fingerprint is a path/size/mtime cache signal, not a byte-level source checksum. Matching test sets make within-quarter score differences interpretable, but Word2Vec and AssetBERT use different objectives, contexts, and training budgets. Even a complete local history remains a 13F adaptation, not a direct FactSet/WRDS paper replication or validated model ranking.

## 8. How this differs from the paper and what comes next

The paper combines FactSet ownership with CRSP/Compustat, aggregates to company level, and tests multiple representation tasks: relative valuation, return comovement, and managed-portfolio similarity. Here we currently have **13F filing-reported, typed security holdings** and have run only the managed-portfolio-similarity adaptation. The paper's original sample, asset split for other tasks, and published ASMP percentages are **not directly comparable** to these local diagnostics. See `docs/PAPER_NOTES.md` for citation and implementation deviations.

The fit-manager-only benchmark corrects the old pilot's all-manager universe filter. The full-history section tests whether its matched-quarter comparison is consistent through time; it does not substitute for a separate validation-manager split for hyperparameters and likelihood temperature, repeated seeds, or equal-compute sensitivity checks. Only after those gates should we move to the paper's valuation and return-comovement tasks. A strategy remains separate.

**Notebook boundary:** none of these charts is a reported publication result. Running the notebook changes no source data or model artifacts. The manager-only result is displayed under its own contract, never pooled with the prefiltered pilot.